# Lesson 5 exercise: your first SFT run

**Goal:** fully fine-tune `SmolLM2-135M-Instruct` with TRL's `SFTTrainer`, and watch its valid-JSON rate on the held-out test set jump.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*. You need the GPU today.
2. Run the cells top to bottom. The whole notebook takes a few minutes.
3. Wherever you see `...` after a `# TODO`, replace it with your code.
4. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup
Installs the course's library versions (about 30 seconds) and defines a tiny `check()` helper.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

## 1 · Task data and the scoring function
The first cell below is the task data from Lesson 4 (`make_rows`). The second defines `score(model, tokenizer, rows)`, which greedily generates a reply for each test row and reports two numbers:
- **valid-JSON rate**: the share of replies that `json.loads` accepts.
- **field accuracy**: the share of `name`, `age` and `city` values that match exactly. A field only counts when the reply parses as JSON, so a badly formatted reply scores 0 on both.

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
def score(model, tokenizer, rows, show=3):
    """Greedy-generate a reply for each row; report valid-JSON rate and field accuracy."""
    model.eval()
    valid, fields_right = 0, 0
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if isinstance(pred, dict):
            fields_right += sum(pred.get(k) == v for k, v in gold.items())
        if i < show:
            print(f"  reply: {reply!r}")
    n = len(rows)
    result = {"valid_json": valid / n, "field_accuracy": fields_right / (3 * n)}
    print(result)
    return result

## 2 · Model, tokenizer and data
The model loads in full precision (`float32`). The T4 has no native bf16 support, and 135M parameters fit easily in fp32.
The training set is the one you built in Lesson 4: same `make_rows` call, same seed. The test rows stay a plain list, which is all `score` needs.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)

train = Dataset.from_list(make_rows(500, "train", seed=0))
test_rows = make_rows(40, "test", seed=1)      # held-out: unseen names, cities and phrasings
print(len(train), "training rows |", len(test_rows), "test rows")

## 3 · Score the base model first
Always measure *before* you train, so you know what fine-tuning changed. Expect replies with the right content in the wrong format, as in Lesson 1.

In [ ]:
print("BEFORE fine-tuning:")
before = score(model, tok, test_rows)

## 4 · Configure the run
`SFTConfig` holds the training settings, like the arguments you pass to Keras `compile` and `fit`.

**TODO 1:** Fill in the two missing values.
- `bf16`: `SFTConfig` turns bf16 on by default. Turn it off for the T4.
- `max_length`: the cap you chose in Lesson 4. The longest row is 91 tokens.

In [ ]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="sft-smollm2-135m",
    num_train_epochs=1,                 # one pass over the 500 rows
    per_device_train_batch_size=8,      # rows per optimizer step
    learning_rate=5e-5,
    logging_steps=10,                   # record loss every 10 steps
    save_strategy="no",                 # no checkpoints on disk today
    report_to="none",                   # no external experiment tracker
    # TODO 1
    bf16=...,
    max_length=...,
)

In [ ]:
# check 1
ok = (isinstance(args, SFTConfig) and args.bf16 is False
      and isinstance(args.max_length, int) and 91 <= args.max_length <= 1024)
check(ok, "Config is T4-safe and no row will be truncated.",
      "bf16 takes a bool. max_length must be an int at least as long as the longest row.")

## 5 · Build the trainer
**TODO 2:** Create an `SFTTrainer` from the `model`, the `args`, the `train` dataset and the tokenizer.
Careful: in TRL 1.x the tokenizer argument is called `processing_class`. Old tutorials say `tokenizer=`, which now raises a `TypeError`.

In [ ]:
# TODO 2
trainer = ...

In [ ]:
# check 2
ok = (isinstance(trainer, SFTTrainer) and trainer.processing_class is tok
      and len(trainer.train_dataset) == len(train))
check(ok, "Trainer ready. It has already templated and tokenized every row.",
      "Pass the model, the config, the training set and the tokenizer. In TRL 1.x the tokenizer keyword isn't `tokenizer=`.")

## 6 · What will the loss count?
Building the trainer already prepared the data: each row now has `input_ids` and `labels`.

**TODO 3:** Keep only the token ids whose label is **not** `-100`, then decode them. This is the Lesson 3 mask, which `SFTTrainer` built for you.

In [ ]:
ex = trainer.train_dataset[0]
print("columns:", list(ex.keys()))
print("tokens:", len(ex["input_ids"]), "| counted by the loss:", sum(l != -100 for l in ex["labels"]))

# TODO 3
counted = ...

print(repr(tok.decode(counted)))

In [ ]:
# check 3
ok = (isinstance(counted, list) and len(counted) == sum(l != -100 for l in ex["labels"])
      and train[0]["completion"][0]["content"] in tok.decode(counted))
check(ok, "Only the completion counts: no system prompt, no user message.",
      "Walk ex['input_ids'] and ex['labels'] side by side and keep the ids whose label isn't -100.")

## 7 · Train
One epoch is 500 rows / 8 per step = 63 optimizer steps. When I tested this notebook, training took about 20 seconds on a laptop GPU (RTX 3050 Ti, bf16 off).
In Colab, a table shows the training loss every 10 steps.

In [ ]:
import time
t0 = time.time()
trainer.train()
print(f"trained in {time.time() - t0:.0f} s")

**Read the log.** `loss` is the cross-entropy on the completion tokens. `mean_token_accuracy` is the share of completion tokens the model predicted exactly. Both are measured on *training* rows.

In [ ]:
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"step {log['step']:>3} | loss {log['loss']:.5f} | mean_token_accuracy {log['mean_token_accuracy']:.3f}")

## 8 · Score the fine-tuned model
Same function, same 40 held-out rows. Only the weights changed.

In [ ]:
print("AFTER fine-tuning:")
after = score(trainer.model, tok, test_rows)
print(f"valid-JSON rate: {before['valid_json']:.3f} -> {after['valid_json']:.3f}")
print(f"field accuracy:  {before['field_accuracy']:.3f} -> {after['field_accuracy']:.3f}")

## 9 · Try your own sentence
**TODO 4:** Write a sentence that contains a name, an age and a city. Try a phrasing unlike the training templates, or a name and city the model never saw.

In [ ]:
# TODO 4
my_text = ...

In [ ]:
# check 4: run the fine-tuned model on your sentence
if isinstance(my_text, str) and my_text.strip():
    messages = [{"role": "user", "content": f"{PROMPT}\n\nText: {my_text}"}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(**inputs, max_new_tokens=40, do_sample=False)
    my_reply = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print("reply:", my_reply)
    try:
        print("✅ valid JSON:", json.loads(my_reply))
    except json.JSONDecodeError:
        print("❌ Not valid JSON. Interesting! Save this sentence: Lesson 8 is about finding failures like this.")
else:
    check(False, "", "Set my_text to a sentence (a string) with a name, an age and a city.")

## Stretch (optional)
1. **Less data.** In section 2, change `500` to `50`. Rerun sections 2–8 (this reloads the base model). Does the score still jump?
2. **Lesson 3, revisited.** Add `completion_only_loss=False` to `SFTConfig`, then rerun sections 4–6. What does the loss count now?
3. **Keep it.** `trainer.save_model("my-extractor")` writes the fine-tuned checkpoint to disk. Lesson 9 shares it on the Hub.